In [10]:
import cv2
import pandas as pd
import torch
from torch import nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50
from tqdm.autonotebook import tqdm
from PIL import Image
from transformers import (
    CLIPProcessor, CLIPModel, DistilBertModel, DistilBertConfig, 
    DistilBertTokenizer, ViTModel, ViTConfig, BertModel, ViTFeatureExtractor
    )


# import os
# import gc
# import numpy as np
# import itertools
# import albumentations as A
# import matplotlib.pyplot as plt

# import timm

$$
\text{Attention}(Q,K,V)=\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

### Tips:
- 数据的大致处理逻辑
  -  image 转成张量
  -  OpenAI CLIP 模型的最大输入长度是 77 个词（包括特殊标记 [CLS] 和 [SEP]）。
  -  坑 图片的评论就是分开放的
  -  注意数据处理时候 tensor格式的形状
  -  resnet和bert成了，下次试试vit

## 1 config

In [2]:
# https://www.kaggle.com/code/moeinshariatnia/openai-clip-simple-implementation/notebook#Image-Encoder
# 这个链接用的resnet50作的解码
# 这个用的transformer加载的模型
# https://blog.csdn.net/weixin_43860330/article/details/129428618
class Arguments(object):
    distil_model_path = '/Users/bowie/Documents/muti-model/distilbert-base-uncased'
    bert_model_path = '/Users/bowie/Documents/muti-model/bert-base-uncased'
    vit_model_path = '/Users/bowie/Documents/muti-model/vit-base-patch16-224'
    resnet50_model_path = '/Users/bowie/Documents/muti-model/resnet-50/pytorch_model.bin'

    text_file = '/Users/bowie/Documents/muti-data/flickr8k/captions.txt'
    
    debug = False
    image_path = "../input/flickr-image-dataset/flickr30k_images/flickr30k_images"
    captions_path = "."
    batch_size = 32
    num_workers = 4
    head_lr = 1e-3
    image_encoder_lr = 1e-4
    text_encoder_lr = 1e-5
    weight_decay = 1e-3
    patience = 1
    factor = 0.8
    epochs = 2
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model_name = 'resnet50'
    image_embedding = 2048
    text_encoder_model = "distilbert-base-uncased"
    text_embedding = 768
    text_tokenizer = "distilbert-base-uncased"
    max_length = 200

    pretrained = True # for both image encoder and text encoder
    trainable = True # for both image encoder and text encoder
    temperature = 1.0

    # image size
    size = 224

    # for projection head; used for both image and text encoders
    num_projection_layers = 1
    projection_dim = 256 
    dropout = 0.1

CFG = Arguments()

## 2 data process
- 直接一步到位 别搞那么多函数
- 切分 loader

In [3]:
df = pd.read_csv(CFG.text_file)

In [4]:
df.head(10)

,image,caption
0,1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg,A girl going into a wooden building .
2,1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a woo...
5,1001773457_577c3a7d70.jpg,A black dog and a spotted dog are fighting
6,1001773457_577c3a7d70.jpg,A black dog and a tri-colored dog playing with...
7,1001773457_577c3a7d70.jpg,A black dog and a white dog with brown spots a...
8,1001773457_577c3a7d70.jpg,Two dogs of different breeds looking at each o...
9,1001773457_577c3a7d70.jpg,Two dogs on pavement moving toward each other .


In [6]:
# new_df = df.groupby("image")["caption"].apply(list).reset_index()

In [7]:
# new_df.head()

,image,caption
0,1000268201_693b08cb0e.jpg,[A child in a pink dress is climbing up a set ...
1,1001773457_577c3a7d70.jpg,"[A black dog and a spotted dog are fighting, A..."
2,1002674143_1b742ab4b8.jpg,[A little girl covered in paint sits in front ...
3,1003163366_44323f5815.jpg,[A man lays on a bench while his dog sits by h...
4,1007129816_e794419615.jpg,[A man in an orange hat starring at something ...


In [8]:
# new_df.tail()

,image,caption
8086,990890291_afc72be141.jpg,[A man does a wheelie on his bicycle on the si...
8087,99171998_7cc800ceef.jpg,"[A group is sitting around a snowy crevasse .,..."
8088,99679241_adc853a5c0.jpg,[A grey bird stands majestically on a beach wh...
8089,997338199_7343367d7f.jpg,"[A person stands near golden walls ., a woman ..."
8090,997722733_0cb5439472.jpg,"[A man in a pink shirt climbs a rock face, A m..."


In [5]:
# 只要前200条数据做测试
# new_df = new_df[:200]
new_df = df[:200]

In [6]:
new_df.tail()

,image,caption
195,1055623002_8195a43714.jpg,A group of four children wearing pajamas have ...
196,1055623002_8195a43714.jpg,A group of kids have a pillow-fight .
197,1055623002_8195a43714.jpg,A group of young children playing pillow fight...
198,1055623002_8195a43714.jpg,Children having a pillow fight .
199,1055623002_8195a43714.jpg,Four children are having a pillow fight .


In [7]:
max_token_length = max(new_df['caption'].apply(len))
max_token_length

108

In [8]:
# from sklearn.model_selection import train_test_split
# train_data, test_data = train_test_split(new_df, test_size=0.2, random_state=42)
# 看情况 要不在别的地方进行分割
train_data, test_data = new_df[:int(len(new_df) * 0.8)], new_df[int(len(new_df) * 0.8):]

In [9]:
# tokenizer = DistilBertTokenizer.from_pretrained(CFG.distil_model_path)
tokenizer = DistilBertTokenizer.from_pretrained(CFG.bert_model_path)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertTokenizer'. 
The class this function is called from is 'DistilBertTokenizer'.
/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### 2.1 resnet数据处理

In [31]:
# 图像转换器
image_transforms = transforms.Compose([
    transforms.Resize((224, 224)),  # 调整到模型需要的尺寸
    transforms.ToTensor(),         # 转换为张量
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # 标准化
])

In [87]:
def process_data(data):
    caption = data['caption']
    image_name = data['image']

    image_path = '/Users/bowie/Documents/muti-data/flickr8k/Images/' + image_name
    # 加载并转换图像
    image = Image.open(image_path).convert("RGB")
    image = image_transforms(image)

    tokenized_captions = tokenizer(
        caption,
        padding="max_length",
        truncation=True,
        max_length=512,
        # return_tensors="pt"
    )

    # 根据索引返回一个样本及其对应的标签
    sample = {
        'image': image,
        'captions': caption,
        'input_ids': torch.tensor(tokenized_captions['input_ids'], requires_grad=False),
        'attention_mask': torch.tensor(tokenized_captions['attention_mask'], requires_grad=False),
    }
    return sample

### 2.2 vit 数据处理

In [11]:
feature_extractor = ViTFeatureExtractor.from_pretrained(CFG.vit_model_path)

# 示例用法
# image = PIL.Image.open("path_to_image.jpg")
# image_tensor = feature_extractor(images=image, return_tensors="pt")["pixel_values"]

/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/transformers/models/vit/feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


In [38]:
# vit 数据处理
def process_data(data):
    caption = data['caption']
    image_name = data['image']

    image_path = '/Users/bowie/Documents/muti-data/flickr8k/Images/' + image_name
    # 加载并转换图像
    image = Image.open(image_path).convert("RGB")
    # image = image_transforms(image)
    image = feature_extractor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)

    tokenized_captions = tokenizer(
        caption,
        padding="max_length",
        truncation=True,
        max_length=512,
        # return_tensors="pt"
    )

    # 根据索引返回一个样本及其对应的标签
    sample = {
        'image': image,
        'captions': caption,
        'input_ids': torch.tensor(tokenized_captions['input_ids'], requires_grad=False),
        'attention_mask': torch.tensor(tokenized_captions['attention_mask'], requires_grad=False),
    }
    return sample

In [37]:
image_pp = '/Users/bowie/Documents/muti-data/flickr8k/Images/1000268201_693b08cb0e.jpg'

image1 = Image.open(image_pp).convert("RGB")
image1 = image_transforms(image1)
print(image1.shape)

image2 = Image.open(image_pp).convert("RGB")
image2 = feature_extractor(images=image2, return_tensors="pt")["pixel_values"]
print(image2.shape)

torch.Size([3, 224, 224])
torch.Size([1, 3, 224, 224])


In [ ]:
# image1 = image1.unsqueeze(0)  # 添加批次维度
# print(image1.shape)  # torch.Size([1, 3, 224, 224])
# image2 = image2.squeeze(0)  # 移除批次维度
# print(image2.shape)  # torch.Size([3, 224, 224])


In [39]:
train_data.iloc[0]

image                              1000268201_693b08cb0e.jpg
caption    A child in a pink dress is climbing up a set o...
Name: 0, dtype: object

In [40]:
train_data_lst = [process_data(train_data.iloc[i]) for i in range(len(train_data))]
print(len(train_data_lst), train_data_lst[:2])

160 [{'image': tensor([[[-0.3569, -0.1294, -0.0902,  ..., -0.9686, -0.9529, -0.9529],
         [-0.3804, -0.1137, -0.0667,  ..., -0.9373, -0.9451, -0.9059],
         [-0.3961, -0.0824, -0.0510,  ..., -0.9373, -0.9451, -0.9216],
         ...,
         [ 0.4588,  0.1765,  0.3412,  ...,  0.6627,  0.2941,  0.2941],
         [ 0.3804,  0.3882,  0.7255,  ...,  0.6471,  0.3176,  0.3176],
         [ 0.6235,  0.6392,  0.4667,  ...,  0.6078,  0.3098,  0.3255]],

        [[-0.3176,  0.0039,  0.0510,  ..., -0.9765, -0.9529, -0.9373],
         [-0.3412,  0.0118,  0.0824,  ..., -0.9294, -0.9216, -0.8353],
         [-0.3804,  0.0353,  0.1059,  ..., -0.9294, -0.8980, -0.8275],
         ...,
         [-0.1529, -0.3725, -0.0431,  ...,  0.7333,  0.4510,  0.4431],
         [-0.2471,  0.0118,  0.3255,  ...,  0.7098,  0.4431,  0.4431],
         [-0.0118,  0.1608, -0.0431,  ...,  0.6784,  0.4431,  0.4431]],

        [[-0.2392, -0.0196, -0.0039,  ..., -0.9765, -0.9686, -0.9608],
         [-0.2784,  0.0118,  0

In [41]:
test_data_lst = [process_data(test_data.iloc[i]) for i in range(len(test_data))]
print(len(test_data_lst), test_data_lst[:2])

40 [{'image': tensor([[[-0.9059, -0.8980, -0.8980,  ..., -0.7961, -0.7961, -0.8039],
         [-0.8980, -0.8980, -0.8980,  ..., -0.8118, -0.8118, -0.8118],
         [-0.8902, -0.8902, -0.8980,  ..., -0.8118, -0.8196, -0.8353],
         ...,
         [-0.6706, -0.7490, -0.6078,  ..., -0.5216, -0.4039, -0.3882],
         [-0.6706, -0.7569, -0.6941,  ..., -0.5922, -0.4667, -0.4431],
         [-0.7255, -0.7098, -0.6235,  ..., -0.6549, -0.6078, -0.4824]],

        [[-0.7725, -0.7569, -0.7490,  ..., -0.6549, -0.6627, -0.6627],
         [-0.7725, -0.7569, -0.7490,  ..., -0.6706, -0.6706, -0.6706],
         [-0.7647, -0.7569, -0.7490,  ..., -0.6706, -0.6784, -0.6941],
         ...,
         [-0.6706, -0.7490, -0.6078,  ..., -0.4745, -0.3569, -0.3490],
         [-0.6706, -0.7569, -0.6941,  ..., -0.5373, -0.4118, -0.3961],
         [-0.7255, -0.7098, -0.6235,  ..., -0.6000, -0.5529, -0.4431]],

        [[-0.5686, -0.5529, -0.5529,  ..., -0.4745, -0.4902, -0.4980],
         [-0.5686, -0.5529, -0.

In [42]:
# 自定义数据集类
class ClipDataset(Dataset):
    def __init__(self, data):
        # 初始化数据和标签
        self.data = data

    def __len__(self):
        # 返回数据集的大小
        return len(self.data)

    def __getitem__(self, idx):
        # 拿到图像的地址
        sample = {
            'image': self.data[idx]['image'],
            'captions': self.data[idx]['captions'],
            'input_ids': self.data[idx]['input_ids'],
            'attention_mask': self.data[idx]['attention_mask'],
        }
        
        return sample

In [43]:
train_dataset = ClipDataset(train_data_lst)
test_dataset = ClipDataset(test_data_lst)

In [44]:
dataset = train_dataset

# 测试前几个样本
for idx in range(3):
    try:
        print(dataset[idx])  # 打印数据，检查输出是否正确
    except Exception as e:
        print(f"Error at index {idx}: {e}")

{'image': tensor([[[-0.3569, -0.1294, -0.0902,  ..., -0.9686, -0.9529, -0.9529],
         [-0.3804, -0.1137, -0.0667,  ..., -0.9373, -0.9451, -0.9059],
         [-0.3961, -0.0824, -0.0510,  ..., -0.9373, -0.9451, -0.9216],
         ...,
         [ 0.4588,  0.1765,  0.3412,  ...,  0.6627,  0.2941,  0.2941],
         [ 0.3804,  0.3882,  0.7255,  ...,  0.6471,  0.3176,  0.3176],
         [ 0.6235,  0.6392,  0.4667,  ...,  0.6078,  0.3098,  0.3255]],

        [[-0.3176,  0.0039,  0.0510,  ..., -0.9765, -0.9529, -0.9373],
         [-0.3412,  0.0118,  0.0824,  ..., -0.9294, -0.9216, -0.8353],
         [-0.3804,  0.0353,  0.1059,  ..., -0.9294, -0.8980, -0.8275],
         ...,
         [-0.1529, -0.3725, -0.0431,  ...,  0.7333,  0.4510,  0.4431],
         [-0.2471,  0.0118,  0.3255,  ...,  0.7098,  0.4431,  0.4431],
         [-0.0118,  0.1608, -0.0431,  ...,  0.6784,  0.4431,  0.4431]],

        [[-0.2392, -0.0196, -0.0039,  ..., -0.9765, -0.9686, -0.9608],
         [-0.2784,  0.0118,  0.0353

In [45]:
train_dataloader = DataLoader(train_dataset,batch_size=4, shuffle=True)
test_dataloader = DataLoader(test_dataset,batch_size=4, shuffle=True)

In [46]:
train_dataset

In [47]:
# type(train_dataloader)
dataset = train_dataloader.dataset
print(len(dataset))  # 检查数据集大小

print(dataset[0]['input_ids'].shape)  # 尝试读取第一条数据
print(dataset[0]['attention_mask'].shape)  # 尝试读取第一条数据
print(dataset[0]['captions'])  # 尝试读取第一条数据
print(dataset[0]['image'].shape)  # 尝试读取第一条数据

160
torch.Size([512])
torch.Size([512])
A child in a pink dress is climbing up a set of stairs in an entry way .
torch.Size([3, 224, 224])


In [48]:
for batch_data in train_dataloader:
    print(len(batch_data), batch_data)
    break

4 {'image': tensor([[[[-0.1529, -0.1373, -0.1608,  ..., -0.4588, -0.3882, -0.4039],
          [-0.0824, -0.0980, -0.1373,  ..., -0.4745, -0.4667, -0.4824],
          [-0.0824, -0.1059, -0.0824,  ..., -0.5137, -0.4980, -0.5216],
          ...,
          [-0.0196,  0.0118, -0.0510,  ..., -0.3333, -0.4510, -0.3569],
          [-0.0980, -0.0745, -0.1373,  ..., -0.3333, -0.3255, -0.2314],
          [-0.0745, -0.1059, -0.2078,  ..., -0.3176, -0.2157, -0.2863]],

         [[ 0.1137,  0.0745,  0.0510,  ..., -0.0745,  0.0118, -0.0510],
          [ 0.1843,  0.1216,  0.0510,  ..., -0.1059, -0.0745, -0.1294],
          [ 0.1765,  0.1216,  0.0902,  ..., -0.1373, -0.1529, -0.2078],
          ...,
          [ 0.2471,  0.3020,  0.2627,  ..., -0.0431, -0.1294, -0.0353],
          [ 0.2157,  0.2157,  0.1765,  ..., -0.0353, -0.0118,  0.0745],
          [ 0.2706,  0.1922,  0.0980,  ...,  0.0039,  0.0902, -0.0118]],

         [[ 0.0588,  0.0510,  0.0196,  ..., -0.6000, -0.5451, -0.5294],
          [ 0.1608

## 3 训练函数搭建

In [104]:

class ResNetEncoder(nn.Module):
    def __init__(self, embed_dim, pretrained=True):
        super(ResNetEncoder, self).__init__()
        # 加载预训练 ResNet 模型
        # /Users/bowie/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
        resnet = resnet50(pretrained=pretrained)

        # 去掉最后的分类层
        self.resnet_base = nn.Sequential(*list(resnet.children())[:-1])  # 去掉全连接层
        # 添加投影层
        self.projection = nn.Linear(resnet.fc.in_features, embed_dim)

    def forward(self, images):
        # 提取 ResNet 特征
        features = self.resnet_base(images)
        features = features.view(features.size(0), -1)  # 展平
        # # 投影到目标维度
        # embeddings = self.projection(features)
        # # 归一化
        # embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        # return embeddings
        return features

# 示例用法
embed_dim = 512
image_encoder = ResNetEncoder(embed_dim=embed_dim, pretrained=True).to(CFG.device)


/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [99]:

class BERTTextEncoder(nn.Module):
    def __init__(self, embed_dim=512):
        super(BERTTextEncoder, self).__init__()
        self.bert = BertModel.from_pretrained(CFG.bert_model_path)  # 加载 BERT 模型
        self.text_projection = nn.Linear(self.bert.config.hidden_size, embed_dim)  # 映射到目标维度

    def forward(self, input_ids, attention_mask):
        # 获取 BERT 模型的输出
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # 使用 [CLS] token 的输出（第一个 token）
        # text_features = output.last_hidden_state[:, 0, :]
        # # 投影到指定维度
        # text_features = self.text_projection(text_features)
        # # 归一化特征
        # text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        # return text_features
        return output


text_encoder = BERTTextEncoder(embed_dim)

A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Pl

In [66]:
text_encoder(input_ids=dataset[0]['input_ids'], attention_mask=dataset[0]['attention_mask'])

tensor([[ 0.0442, -0.0267, -0.0230, -0.0326,  0.0321,  0.0034, -0.0265,  0.0086,
         -0.0259,  0.0084, -0.0160,  0.0249, -0.0004, -0.1256,  0.0376, -0.0742,
         -0.0044,  0.0032,  0.0670, -0.0556, -0.0502, -0.0253, -0.0115, -0.0049,
         -0.0145,  0.0680, -0.0893,  0.0421,  0.0043,  0.0273, -0.0096,  0.0343,
          0.0462,  0.0052, -0.0444, -0.0017,  0.0708,  0.0171,  0.1264,  0.0514,
          0.0140,  0.0306, -0.0724,  0.0215,  0.0111,  0.0145,  0.1024,  0.0248,
          0.0098,  0.0237,  0.0168,  0.0361,  0.0127, -0.0293,  0.0537, -0.0034,
          0.0456,  0.0037, -0.0071,  0.0125,  0.0048,  0.0274,  0.0600,  0.0409,
         -0.0662,  0.0338,  0.0196,  0.0220, -0.0344, -0.0179,  0.0121,  0.0161,
         -0.0036, -0.0129,  0.0160,  0.0682,  0.0651,  0.0074, -0.0653, -0.0419,
         -0.0148,  0.0531,  0.0456,  0.0226, -0.0134,  0.0165,  0.0662, -0.0289,
          0.0087,  0.0124, -0.0275, -0.0141,  0.0362,  0.0100, -0.0049, -0.0129,
          0.0182, -0.1230, -

In [27]:
def contrastive_loss(similarity_matrix):
    labels = torch.arange(similarity_matrix.size(0)).to(similarity_matrix.device)
    loss_text_to_image = nn.CrossEntropyLoss()(similarity_matrix, labels)
    loss_image_to_text = nn.CrossEntropyLoss()(similarity_matrix.T, labels)
    return (loss_text_to_image + loss_image_to_text) / 2


In [105]:
class CLIPModelNet(nn.Module):
    def __init__(self, embed_dim):
        super(CLIPModelNet, self).__init__()
        
        # 文本编码器（DistilBert）
        # self.text_encoder = DistilBertModel.from_pretrained(CFG.distil_model_path)
        # self.text_encoder = BertModel.from_pretrained(CFG.distil_model_path)
        self.text_encoder = text_encoder
        self.text_projection = nn.Linear(768, embed_dim)
        
        # 图像编码器（如 ResNet 或 ViT）
        self.image_encoder = image_encoder
        # self.image_projection = nn.Linear(self.image_encoder.fc.out_features, embed_dim)
        # 下面两个是resnet的处理方式
        # self.image_projection = nn.Linear(self.image_encoder.resnet_base[-1][-1].bn3.num_features, embed_dim)
        self.image_projection = nn.Linear(2048, embed_dim)  # ResNet50 的输出特征维度是 2048

        # 归一化层
        self.logit_scale = nn.Parameter(torch.ones([]) * 0.07)
    
    def forward(self, input_ids, attention_mask, images):
        # 文本编码
        # print(self.text_encoder.config)
        text_features = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_features = text_features.last_hidden_state
        text_features = text_features[:, 0, :]  # 取 [CLS] token
        text_features = self.text_projection(text_features)
        
        # 图像编码
        image_features = self.image_encoder(images)
        image_features = self.image_projection(image_features)
        
        # 归一化
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        
        return text_features, image_features, self.logit_scale

    def compute_similarity(self, text_features, image_features):
        # 计算相似度
        return self.logit_scale.exp() * torch.matmul(text_features, image_features.T)

In [106]:
# 初始化模型
model = CLIPModelNet(embed_dim=512).to(CFG.device)

In [107]:
# 优化器
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

In [108]:
# 训练循环
for epoch in range(2):
    model.train()
    total_loss = 0

    for batch in train_dataloader:
        # images, input_ids, attention_mask = [x.to(CFG.device) for x in batch]
        images = batch['image'].to(CFG.device)
        input_ids = batch['input_ids'].to(CFG.device)
        attention_mask = batch['attention_mask'].to(CFG.device)
        
        # 前向传播
        # print(input_ids.shape, attention_mask.shape)
        text_features, image_features, logit_scale = model(input_ids, attention_mask, images)
        similarity_matrix = model.compute_similarity(text_features, image_features)
        
        # 计算损失
        loss = contrastive_loss(similarity_matrix)
        total_loss += loss.item()
        
        # 反向传播与优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f"Epoch {epoch + 1}/{2}, Loss: {total_loss:.4f}")


Epoch 1/2, Loss: 48.4580
Epoch 2/2, Loss: 39.8628


## another 3 训练函数搭建
vit and distil bert

In [23]:
class ViTEncoder(nn.Module):
    def __init__(self, embed_dim):
        super(ViTEncoder, self).__init__()
        # 加载预训练的 ViT 模型
        self.vit = ViTModel.from_pretrained(CFG.vit_model_path)
        # 添加投影层
        self.projection = nn.Linear(in_features=self.vit.config.hidden_size, embed_dim)

    def forward(self, images):
        # ViT 特征提取
        outputs = self.vit(pixel_values=images)
        features = outputs.pooler_output  # 使用 pooler_output 作为全局特征
        # 投影到目标维度
        # embeddings = self.projection(features)
        # # 归一化
        # embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        # return embeddings
        return features

# 示例用法
embed_dim = 512
image_encoder1 = ViTEncoder(embed_dim=embed_dim).to(CFG.device)


Some weights of ViTModel were not initialized from the model checkpoint at /Users/bowie/Documents/muti-model/vit-base-patch16-224 and are newly initialized: ['vit.pooler.dense.bias', 'vit.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [24]:
class DistilBERTTextEncoder(nn.Module):
    def __init__(self, embed_dim=512):
        super(DistilBERTTextEncoder, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained(CFG.distil_model_path)  # 加载 DistilBERT 模型
        self.text_projection = nn.Linear(self.distilbert.config.hidden_size, embed_dim)  # 映射到目标维度

    def forward(self, input_ids, attention_mask):
        # 获取 DistilBERT 模型的输出
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        # 使用 [CLS] token 的输出（第一个 token）
        # text_features = outputs.last_hidden_state[:, 0, :]
        # # 投影到指定维度
        # text_features = self.text_projection(text_features)
        # # 归一化特征
        # text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        # return text_features
        return outputs


text_encoder1 = DistilBERTTextEncoder(embed_dim)

In [50]:
class CLIPModelNet(nn.Module):
    def __init__(self, embed_dim):
        super(CLIPModelNet, self).__init__()
        
        # 文本编码器（DistilBert）
        # self.text_encoder = DistilBertModel.from_pretrained(CFG.distil_model_path)
        # self.text_encoder = BertModel.from_pretrained(CFG.distil_model_path)
        self.text_encoder = text_encoder1
        self.text_projection = nn.Linear(768, embed_dim)
        
        # 图像编码器（如 ResNet 或 ViT）
        self.image_encoder = image_encoder1
        # self.image_projection = nn.Linear(self.image_encoder.fc.out_features, embed_dim)
        self.image_projection = nn.Linear(768, embed_dim)  # ResNet50 的输出特征维度是 2048

        # 归一化层
        self.logit_scale = nn.Parameter(torch.ones([]) * 0.07)
    
    def forward(self, input_ids, attention_mask, images):
        # 文本编码
        # print(self.text_encoder.config)
        text_features = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_features = text_features.last_hidden_state
        text_features = text_features[:, 0, :]  # 取 [CLS] token
        text_features = self.text_projection(text_features)
        
        # 图像编码
        image_features = self.image_encoder(images)
        image_features = self.image_projection(image_features)
        
        # 归一化
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        
        return text_features, image_features, self.logit_scale

    def compute_similarity(self, text_features, image_features):
        # 计算相似度
        return self.logit_scale.exp() * torch.matmul(text_features, image_features.T)

In [26]:
text_encoder1(input_ids=dataset[0]['input_ids'], attention_mask=dataset[0]['attention_mask'])

BaseModelOutput(last_hidden_state=tensor([[[-0.2001, -0.3992, -0.0373,  ...,  0.0144,  0.2265,  0.3925],
         [-0.3384,  0.0443, -0.0466,  ..., -0.3810,  0.1735,  0.2331],
         [-0.5300,  0.0229,  0.1030,  ..., -0.6186,  0.2054, -0.0256],
         ...,
         [ 0.1194, -0.0385,  0.1286,  ..., -0.0874, -0.1371,  0.1309],
         [ 0.1310, -0.1760,  0.0915,  ...,  0.0915, -0.1798,  0.1215],
         [ 0.1304, -0.2286,  0.1505,  ...,  0.0463, -0.1948, -0.0148]]],
       grad_fn=<NativeLayerNormBackward0>), hidden_states=None, attentions=None)

In [51]:
# 初始化模型
model = CLIPModelNet(embed_dim=512).to(CFG.device)

In [29]:
# 优化器
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

In [52]:
# 训练循环
for epoch in range(2):
    model.train()
    total_loss = 0

    for batch in train_dataloader:
        # images, input_ids, attention_mask = [x.to(CFG.device) for x in batch]
        images = batch['image'].to(CFG.device)
        input_ids = batch['input_ids'].to(CFG.device)
        attention_mask = batch['attention_mask'].to(CFG.device)
        
        # 前向传播
        # print(input_ids.shape, attention_mask.shape)
        text_features, image_features, logit_scale = model(input_ids, attention_mask, images)
        similarity_matrix = model.compute_similarity(text_features, image_features)
        
        # 计算损失
        loss = contrastive_loss(similarity_matrix)
        total_loss += loss.item()
        
        # 反向传播与优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f"Epoch {epoch + 1}/{2}, Loss: {total_loss:.4f}")


Epoch 1/2, Loss: 47.3754
Epoch 2/2, Loss: 36.2116


In [ ]:
def train_epoch(model, train_loader, optimizer, lr_scheduler, step):
    loss_meter = AvgMeter()
    tqdm_object = tqdm(train_loader, total=len(train_loader))
    for batch in tqdm_object:
        batch = {k: v.to(CFG.device) for k, v in batch.items() if k != "caption"}
        loss = model(batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step == "batch":
            lr_scheduler.step()

        count = batch["image"].size(0)
        loss_meter.update(loss.item(), count)

        tqdm_object.set_postfix(train_loss=loss_meter.avg, lr=get_lr(optimizer))
    return loss_meter


def valid_epoch(model, valid_loader):
    loss_meter = AvgMeter()

    tqdm_object = tqdm(valid_loader, total=len(valid_loader))
    for batch in tqdm_object:
        batch = {k: v.to(CFG.device) for k, v in batch.items() if k != "caption"}
        loss = model(batch)

        count = batch["image"].size(0)
        loss_meter.update(loss.item(), count)

        tqdm_object.set_postfix(valid_loss=loss_meter.avg)
    return loss_meter



## 主函数训练

In [ ]:

def main():
    train_df, valid_df = make_train_valid_dfs()
    tokenizer = DistilBertTokenizer.from_pretrained(CFG.text_tokenizer)
    train_loader = build_loaders(train_df, tokenizer, mode="train")
    valid_loader = build_loaders(valid_df, tokenizer, mode="valid")


    model = CLIPModel().to(CFG.device)
    params = [
        {"params": model.image_encoder.parameters(), "lr": CFG.image_encoder_lr},
        {"params": model.text_encoder.parameters(), "lr": CFG.text_encoder_lr},
        {"params": itertools.chain(
            model.image_projection.parameters(), model.text_projection.parameters()
        ), "lr": CFG.head_lr, "weight_decay": CFG.weight_decay}
    ]
    optimizer = torch.optim.AdamW(params, weight_decay=0.)
    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=CFG.patience, factor=CFG.factor
    )
    step = "epoch"

    best_loss = float('inf')
    for epoch in range(CFG.epochs):
        print(f"Epoch: {epoch + 1}")
        model.train()
        train_loss = train_epoch(model, train_loader, optimizer, lr_scheduler, step)
        model.eval()
        with torch.no_grad():
            valid_loss = valid_epoch(model, valid_loader)
        
        if valid_loss.avg < best_loss:
            best_loss = valid_loss.avg
            torch.save(model.state_dict(), "best.pt")
            print("Saved Best Model!")
        
        lr_scheduler.step(valid_loss.avg)

In [ ]:
def find_matches(model, image_embeddings, query, image_filenames, n=9):
    tokenizer = DistilBertTokenizer.from_pretrained(CFG.text_tokenizer)
    encoded_query = tokenizer([query])
    batch = {
        key: torch.tensor(values).to(CFG.device)
        for key, values in encoded_query.items()
    }
    with torch.no_grad():
        text_features = model.text_encoder(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]
        )
        text_embeddings = model.text_projection(text_features)
    
    image_embeddings_n = F.normalize(image_embeddings, p=2, dim=-1)
    text_embeddings_n = F.normalize(text_embeddings, p=2, dim=-1)
    dot_similarity = text_embeddings_n @ image_embeddings_n.T
    
    values, indices = torch.topk(dot_similarity.squeeze(0), n * 5)
    matches = [image_filenames[idx] for idx in indices[::5]]
    
    _, axes = plt.subplots(3, 3, figsize=(10, 10))
    for match, ax in zip(matches, axes.flatten()):
        image = cv2.imread(f"{CFG.image_path}/{match}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        ax.imshow(image)
        ax.axis("off")
    
    plt.show()

In [ ]:
find_matches(model, 
             image_embeddings,
             query="one dog sitting on the grass",
             image_filenames=valid_df['image'].values,
             n=9)